In [11]:
import concurrent.futures
import time

In [12]:
def task(n, msg:str = ''):
	"""A sample task that returns n / 10."""
	t0 = time.time()
	
	time.sleep(n % 3)
	if n < 0:
		raise ValueError(f'Invalid value: {n}.  Must use non-negative numbers.')
	elif msg == "Break":
		raise Exception(f'Break requested for value {n}')
	
	else:
		s = f'{n / 10} \t{msg}\t({time.time() - t0:0.4f} sec)'
		return s


def done_callback(future):
	"""Callback function to process the completed future."""
	if future.cancelled():
		print(f"Future was cancelled.")
	elif future.exception():
		error = future.exception()
		print(f"Future raised an exception: {error}")
	else:
		result = future.result()
		print(f"Future returned result: {result}")

In [13]:
sep = '......................'
t0 = time.time()
print(f'Main thread begins\n{sep}')

# Create an executor and submit a task in context of an executor
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
	fs = {executor.submit(task, i, 'Break'): i for i in [6, 7]}
	fs = fs | {executor.submit(task, i, 'Big numbers!'): i for i in [25, 50]}
	fs = fs | {executor.submit(task, n=i): i for i in range(-2,11)}
	for f in fs:
		# Attach the callback
		f.add_done_callback(done_callback)

# Wait a moment so some, but not all futures complete	
time.sleep(1)

# Try cancelling anything still running - not working, probably because my callable is too simple
for future in fs:
	future.cancel()

# Write out results, along with the initial parameter (or error if encountered)
print(f'{sep}\nWrite results\n{sep}')
for future in concurrent.futures.as_completed(fs):

	# Print results for those without exceptions, error message for those that failed
	if not future.exception():
		print(fs[future], ' -\t', future.result())
	else:
		print(fs[future], ' -\t', future.exception())
	
# Final status
print(f'{sep}\nMain thread ends\n\tTotal: {(time.time() - t0):0.3f} sec')

Main thread begins
......................
Future raised an exception: Break requested for value 6
Future returned result: 0.0 		(0.0000 sec)
Future returned result: 0.3 		(0.0001 sec)
Future returned result: 2.5 	Big numbers!	(1.0001 sec)Future raised an exception: Break requested for value 7
Future raised an exception: Invalid value: -2.  Must use non-negative numbers.
Future returned result: 0.6 		(0.0000 sec)

Future returned result: 0.1 		(1.0007 sec)
Future returned result: 0.9 		(0.0000 sec)
Future returned result: 0.4 		(1.0005 sec)
Future returned result: 5.0 	Big numbers!	(2.0002 sec)Future raised an exception: Invalid value: -1.  Must use non-negative numbers.
Future returned result: 0.2 		(2.0004 sec)

Future returned result: 0.7 		(1.0009 sec)
Future returned result: 1.0 		(1.0003 sec)
Future returned result: 0.5 		(2.0005 sec)
Future returned result: 0.8 		(2.0022 sec)
......................
Write results
......................
6  -	 Break requested for value 6
8  -	 0.8 	